In [28]:
#r "nuget:ScottPlot,5.0.21"
#r "task14/bin/Debug/net10.0/task14.dll"

using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using ScottPlot;
using task14;

double a = -100, b = 100;
double targetPrecision = 1e-4;
int warmupRuns = 200;
int measureRuns = 20;
int maxThreads = Environment.ProcessorCount * 2;

Func<double, double> f = Math.Sin;
double exactValue = 0.0;

double[] stepCandidates = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };

double optimalStep = 0;
int optimalThreadsForStep = 1;
double bestParallelTimeForOptimalStep = 0;
double bestSingleTimeForOptimalStep = 0;
double bestImprovement = 0;
bool foundOptimalStep = false;

var allStepData = new List<(double step, int threads, double singleTime, double parallelTime, double improvement)>();
var bestAverageTimes = new List<double>();
var threadCounts = new List<int>();

foreach (var step in stepCandidates)
{
    double val = OneThread.Solve(a, b, f, step);
    double error = Math.Abs(val - exactValue);
    
    if (error > targetPrecision)
    {
        Console.WriteLine($"\nШаг {step:E1}: точность {error:E3} (недостаточно, нужно < {targetPrecision:E1})");
        continue;
    }

    var swSingle = new Stopwatch();
    double singleTotal = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        swSingle.Restart();
        OneThread.Solve(a, b, f, step);
        swSingle.Stop();
        if (run >= warmupRuns) singleTotal += swSingle.Elapsed.TotalMilliseconds;
    }
    double singleTime = singleTotal / measureRuns;
    
    Console.WriteLine($"Шаг {step:E1}: точность = {error:E5}, однопоточное время = {singleTime:F3} мс");
    Console.WriteLine($"Тестирование потоков от 1 до {maxThreads}");
    
    double bestParallelTime = double.MaxValue;
    int bestThreads = 1;
    
    for (int threads = 2; threads <= maxThreads; threads += 2)
    {
        var sw = new Stopwatch();
        double totalTime = 0;
        
        for (int run = 0; run < warmupRuns + measureRuns; run++)
        {
            sw.Restart();
            DefiniteIntegral.Solve(a, b, f, step, threads);
            sw.Stop();
            if (run >= warmupRuns) totalTime += sw.Elapsed.TotalMilliseconds;
        }
        
        double avgTime = totalTime / measureRuns;
        
        if (avgTime < bestParallelTime)
        {
            bestParallelTime = avgTime;
            bestThreads = threads;
        }
    }
    
    double improvement = (singleTime - bestParallelTime) / singleTime * 100;
    
    allStepData.Add((step, bestThreads, singleTime, bestParallelTime, improvement));
    bestAverageTimes.Add(bestParallelTime);
    threadCounts.Add(bestThreads);
    
    Console.WriteLine($"Лучший результат: {bestThreads} потоков, время = {bestParallelTime:F3} мс");
    Console.WriteLine($"Прирост производительности: {improvement:F1}%");
    
    optimalStep = step;
    optimalThreadsForStep = bestThreads;
    bestParallelTimeForOptimalStep = bestParallelTime;
    bestSingleTimeForOptimalStep = singleTime;
    bestImprovement = improvement;
    
    if (improvement >= 15)
    {
        Console.WriteLine($"\nДостигнут прирост >15% на шаге {step:E1}! Это оптимальный шаг.");
        foundOptimalStep = true;
        break;
    }
    else
    {
        Console.WriteLine($"Прирост {improvement:F1}% < 15%, переходим к следующему шагу...");
    }
}

if (!foundOptimalStep)
{
    Console.WriteLine($"\nНи на одном шаге не достигнут прирост 15%. Используем последний проверенный шаг {optimalStep:E1} с приростом {bestImprovement:F1}%");
}

var threadTimes = new List<(int threads, double time)>();

foreach (var step in stepCandidates)
{
    double val = OneThread.Solve(a, b, f, step);
    double error = Math.Abs(val - exactValue);
    
    if (error > targetPrecision)
    {
        Console.WriteLine($"\nШаг {step:E1}: точность {error:E3} (недостаточно, нужно < {targetPrecision:E1})");
        continue;
    }

    var swSingle = new Stopwatch();
    double singleTotal = 0;
    for (int run = 0; run < warmupRuns + measureRuns; run++)
    {
        swSingle.Restart();
        OneThread.Solve(a, b, f, step);
        swSingle.Stop();
        if (run >= warmupRuns) singleTotal += swSingle.Elapsed.TotalMilliseconds;
    }
    double singleTime = singleTotal / measureRuns;
    
    Console.WriteLine($"Шаг {step:E1}: точность = {error:E5}, однопоточное время = {singleTime:F3} мс");
    Console.WriteLine($"Тестирование потоков от 1 до {maxThreads}");
    
    double bestParallelTime = double.MaxValue;
    int bestThreads = 1;
    
    var currentStepTimes = new List<(int threads, double time)>();
    currentStepTimes.Add((1, singleTime));
    
    for (int threads = 2; threads <= maxThreads; threads += 2)
    {
        var sw = new Stopwatch();
        double totalTime = 0;
        
        for (int run = 0; run < warmupRuns + measureRuns; run++)
        {
            sw.Restart();
            DefiniteIntegral.Solve(a, b, f, step, threads);
            sw.Stop();
            if (run >= warmupRuns) totalTime += sw.Elapsed.TotalMilliseconds;
        }
        
        double avgTime = totalTime / measureRuns;
        currentStepTimes.Add((threads, avgTime));
        
        if (avgTime < bestParallelTime)
        {
            bestParallelTime = avgTime;
            bestThreads = threads;
        }
    }
    
    double improvement = (singleTime - bestParallelTime) / singleTime * 100;
    
    allStepData.Add((step, bestThreads, singleTime, bestParallelTime, improvement));
    
    Console.WriteLine($"Лучший результат: {bestThreads} потоков, время = {bestParallelTime:F3} мс");
    Console.WriteLine($"Прирост производительности: {improvement:F1}%");
    
    optimalStep = step;
    optimalThreadsForStep = bestThreads;
    bestParallelTimeForOptimalStep = bestParallelTime;
    bestSingleTimeForOptimalStep = singleTime;
    bestImprovement = improvement;
    threadTimes = currentStepTimes;
    
    if (improvement >= 15)
    {
        Console.WriteLine($"\nДостигнут прирост >15% на шаге {step:E1}! Это оптимальный шаг.");
        foundOptimalStep = true;
        break;
    }
    else
    {
        Console.WriteLine($"Прирост {improvement:F1}% < 15%, переходим к следующему шагу...");
    }
}

var plt = new ScottPlot.Plot();

double[] plotX = threadTimes.Select(t => t.time).ToArray();
double[] plotY = threadTimes.Select(t => (double)t.threads).ToArray();

var scatter = plt.Add.Scatter(plotX, plotY);
scatter.LineWidth = 3;
scatter.MarkerSize = 10;
scatter.Color = ScottPlot.Color.FromHex("#1e3a8a");

plt.Title($"Зависимость производительности от количества потоков (шаг {optimalStep:E1})");
plt.XLabel("Количество потоков");
plt.YLabel("Время вычисления (мс)");

string timestamp = DateTime.Now.Ticks.ToString();
string fileName = $"threads_performance_{optimalStep:E1}_{timestamp}.png";

if (File.Exists(fileName))
    File.Delete(fileName);

plt.SavePng(fileName, 700, 500);
Console.WriteLine($"График сохранен: {fileName}");

Console.WriteLine($"ИТОГОВЫЙ ОПТИМАЛЬНЫЙ ШАГ: {optimalStep:E1}");
Console.WriteLine($"   Точность: < {targetPrecision:E1}");
Console.WriteLine($"   Потоков: {optimalThreadsForStep}");
Console.WriteLine($"   Время (1 поток): {bestSingleTimeForOptimalStep:F3} мс");
Console.WriteLine($"   Время ({optimalThreadsForStep} потоков): {bestParallelTimeForOptimalStep:F3} мс");
Console.WriteLine($"   Прирост: {bestImprovement:F1}%");
Console.WriteLine($"   Критерий 15%: {(bestImprovement >= 15 ? "ВЫПОЛНЕН" : "НЕ ВЫПОЛНЕН")}\n");

string resultFilePath = Path.Combine(Directory.GetCurrentDirectory(), "optimal_params.txt");
string results = 
$@"РЕЗУЛЬТАТЫ ИССЛЕДОВАНИЯ ОПТИМАЛЬНЫХ ПАРАМЕТРОВ
Дата: {DateTime.Now:yyyy-MM-dd HH:mm:ss}

1. ВЫБРАННЫЙ ШАГ ИНТЕГРИРОВАНИЯ:
   Шаг: {optimalStep:E1}
   Обеспечиваемая точность: < {targetPrecision:E1}
   Критерий выбора: первый шаг, где многопоточный прирост ≥ 15%

2. ОПТИМАЛЬНЫЕ ПАРАМЕТРЫ:
   Шаг: {optimalStep:E1}
   Количество потоков: {optimalThreadsForStep}
   Время выполнения (многопот.): {bestParallelTimeForOptimalStep:F3} мс
   Время выполнения (однопот.): {bestSingleTimeForOptimalStep:F3} мс
   Прирост производительности: {bestImprovement:F1}%
   Критерий (ускорение ≥ 15%): {(bestImprovement >= 15 ? "ВЫПОЛНЕН" : "НЕ ВЫПОЛНЕН")}
";

File.WriteAllText(resultFilePath, results);
Console.WriteLine($"\nРезультаты сохранены в файл: {resultFilePath}");

Installed Packages ScottPlot, 5.0.21

Шаг 1.0E-001: точность = 5.04458E-015, однопоточное время = 0.049 мс
Тестирование потоков от 1 до 16
Лучший результат: 2 потоков, время = 0.304 мс
Прирост производительности: -515.8%
Прирост -515.8% < 15%, переходим к следующему шагу...
Шаг 1.0E-002: точность = 1.49195E-014, однопоточное время = 0.692 мс
Тестирование потоков от 1 до 16
Лучший результат: 2 потоков, время = 0.619 мс
Прирост производительности: 10.5%
Прирост 10.5% < 15%, переходим к следующему шагу...
Шаг 1.0E-003: точность = 2.34296E-016, однопоточное время = 5.371 мс
Тестирование потоков от 1 до 16
Лучший результат: 6 потоков, время = 2.515 мс
Прирост производительности: 53.2%

Достигнут прирост >15% на шаге 1.0E-003! Это оптимальный шаг.
Шаг 1.0E-001: точность = 5.04458E-015, однопоточное время = 0.070 мс
Тестирование потоков от 1 до 16
Лучший результат: 2 потоков, время = 0.346 мс
Прирост производительности: -392.9%
Прирост -392.9% < 15%, переходим к следующему шагу...
Шаг 1.0E-002: точность = 1.49195E-014, однопоточн